In [ ]:
import json
import platform
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11 or later is required.")

if any(
    name == "eurusd_research" or name.startswith("eurusd_research.")
    for name in sys.modules
):
    raise RuntimeError(
        "Restart the kernel, then run this notebook from Cell 2. "
        "This prevents stale project imports."
    )

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / ".git").exists()),
    None,
)

if REPO_ROOT is None:
    REPO_ROOT = Path(
        input("Full path to the existing project repository: ").strip().strip('"')
    ).expanduser().resolve()

if not (REPO_ROOT / ".git").exists():
    raise FileNotFoundError("The selected directory is not an existing Git checkout.")

example_path = REPO_ROOT / "configs/research_config.example.json"
local_path = REPO_ROOT / "configs/research_config.local.json"

if not example_path.is_file() or not local_path.is_file():
    raise FileNotFoundError("Notebook 00 configuration files are required.")

print("Repository:", REPO_ROOT)
print("Python:", platform.python_version())

planned_paths = [
    "pyproject.toml",
    "src/eurusd_research/__init__.py",
    "src/eurusd_research/config.py",
    "src/eurusd_research/quality.py",
    "src/eurusd_research/data.py",
    "tests/test_alignment.py",
    "tests/test_partition_consistency.py",
    "data/sample/synthetic_quotes.csv",
]

for relative in planned_paths:
    path = REPO_ROOT / relative
    print("EXISTS" if path.exists() else "MISSING", relative)

new_files = {}


def create_missing_files(contents):
    """Preflight every target before creating any files."""
    conflicts = []

    for relative, text in contents.items():
        path = REPO_ROOT / relative

        if not path.resolve().is_relative_to(REPO_ROOT):
            raise ValueError(f"Target escapes the repository: {relative}")

        if path.exists():
            if not path.is_file() or path.read_text(encoding="utf-8-sig") != text:
                conflicts.append(relative)

    if conflicts:
        raise FileExistsError(
            "Existing files differ; nothing was written. Return this list:\n"
            + "\n".join(conflicts)
        )

    for relative, text in contents.items():
        path = REPO_ROOT / relative

        if path.exists():
            print("MATCHES", relative)
        else:
            path.parent.mkdir(parents=True, exist_ok=True)
            with path.open("x", encoding="utf-8", newline="\n") as handle:
                handle.write(text)
            print("CREATED", relative)

In [ ]:
new_files["pyproject.toml"] = '''\
[build-system]
requires = ["setuptools"]
build-backend = "setuptools.build_meta"

[project]
name = "eurusd-fair-value-liquidity"
version = "0.1.0"
description = "Research on EURUSD quote predictability and displayed liquidity"
requires-python = ">=3.11"
dependencies = [
    "numpy",
    "pandas",
    "pyarrow",
    "duckdb",
    "matplotlib",
    "tzdata",
    "pytz",
]

[project.optional-dependencies]
modeling = ["scikit-learn"]
test = ["pytest"]
notebooks = ["jupyterlab", "ipykernel"]

[tool.setuptools]
package-dir = {"" = "src"}

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-ra"
'''

new_files["src/eurusd_research/__init__.py"] = '''\
"""EURUSD fair-value and liquidity research utilities."""

__version__ = "0.1.0"
'''

new_files["src/eurusd_research/config.py"] = '''\
"""Load shared configuration with optional machine-specific overrides."""

import json
from pathlib import Path


def _merge(base, override):
    result = dict(base)
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = _merge(result[key], value)
        else:
            result[key] = value
    return result


def load_config(repository_root):
    root = Path(repository_root).resolve()
    example = root / "configs/research_config.example.json"
    local = root / "configs/research_config.local.json"

    config = json.loads(example.read_text(encoding="utf-8-sig"))
    if local.exists():
        config = _merge(
            config,
            json.loads(local.read_text(encoding="utf-8-sig")),
        )

    raw = Path(config["raw_data_dir"]).expanduser()
    config["raw_data_dir"] = str(
        raw.resolve() if raw.is_absolute() else (root / raw).resolve()
    )
    return config
'''

print("Package and configuration module contents prepared.")

In [ ]:
new_files["src/eurusd_research/quality.py"] = r'''\
"""Vectorized parsing and quote-quality flags."""

import numpy as np
import pandas as pd

EXPECTED_COLUMNS = ["Time (EET)", "Ask", "Bid", "AskVolume", "BidVolume"]


def normalise_quotes(
    raw,
    *,
    source_name,
    source_sha256,
    first_row,
    previous_utc,
    timezone,
    timestamp_format,
    pip_size,
    gap_seconds,
    extreme_spread_pips,
):
    if list(raw.columns) != EXPECTED_COLUMNS:
        raise ValueError(f"Unexpected CSV schema: {list(raw.columns)}")

    frame = raw.rename(columns={
        "Time (EET)": "timestamp_source",
        "Ask": "ask_raw",
        "Bid": "bid_raw",
        "AskVolume": "ask_size_raw",
        "BidVolume": "bid_size_raw",
    }).reset_index(drop=True).fillna("")

    source_text = frame["timestamp_source"].str.strip()
    parsed = pd.to_datetime(
        source_text, format=timestamp_format, errors="coerce", exact=True
    )
    utc = parsed.dt.tz_localize(
        timezone, ambiguous="NaT", nonexistent="NaT"
    ).dt.tz_convert("UTC")

    frame["timestamp_utc"] = utc
    frame["source_file"] = source_name
    frame["source_sha256"] = source_sha256
    frame["source_row"] = np.arange(
        first_row, first_row + len(frame), dtype=np.int64
    )

    for name in ("ask", "bid", "ask_size", "bid_size"):
        frame[name] = pd.to_numeric(
            frame[f"{name}_raw"].str.strip(), errors="coerce"
        ).to_numpy(dtype="float64", na_value=np.nan)

    ask, bid = frame["ask"], frame["bid"]
    ask_size, bid_size = frame["ask_size"], frame["bid_size"]
    valid_price = (
        np.isfinite(ask) & np.isfinite(bid) & ask.gt(0) & bid.gt(0)
    )
    ask_blank = frame["ask_size_raw"].str.strip().eq("")
    bid_blank = frame["bid_size_raw"].str.strip().eq("")

    frame["flag_missing_timestamp"] = source_text.eq("")
    frame["flag_unparseable_timestamp"] = source_text.ne("") & parsed.isna()
    frame["flag_unresolved_local_time"] = parsed.notna() & utc.isna()
    frame["flag_invalid_price"] = ~valid_price
    frame["flag_nonpositive_price"] = (
        (np.isfinite(ask) & ask.le(0))
        | (np.isfinite(bid) & bid.le(0))
    )
    frame["flag_crossed"] = valid_price & ask.lt(bid)
    frame["flag_locked"] = valid_price & ask.eq(bid)
    frame["flag_missing_size"] = ask_blank | bid_blank
    frame["flag_invalid_size"] = (
        (~np.isfinite(ask_size) & ~ask_blank)
        | (~np.isfinite(bid_size) & ~bid_blank)
    )
    frame["flag_zero_size"] = ask_size.eq(0) | bid_size.eq(0)
    frame["flag_negative_size"] = (
        (np.isfinite(ask_size) & ask_size.lt(0))
        | (np.isfinite(bid_size) & bid_size.lt(0))
    )
    frame["flag_zero_total_size"] = (
        np.isfinite(ask_size) & np.isfinite(bid_size)
        & ask_size.ge(0) & bid_size.ge(0)
        & (ask_size + bid_size).eq(0)
    )

    frame["spread_pips"] = ((ask - bid) / pip_size).where(valid_price)
    frame["flag_extreme_spread"] = frame["spread_pips"].gt(
        extreme_spread_pips
    )

    previous = utc.shift()
    if len(frame) and previous_utc is not None:
        previous.iloc[0] = pd.Timestamp(previous_utc)

    difference = (utc - previous).dt.total_seconds()
    frame["source_gap_seconds"] = difference
    frame["flag_backward_timestamp"] = difference.lt(0)
    frame["flag_adjacent_repeated_timestamp"] = (
        utc.notna() & previous.notna() & utc.eq(previous)
    )
    frame["flag_source_gap"] = difference.gt(gap_seconds)

    return frame
'''.removeprefix("\\\n")

print("Quality module contents prepared.")

In [ ]:
data_helpers = r'''\
"""Bounded CSV ingestion, integrity manifests, and dataset diagnostics."""

import hashlib
import json
import os
import platform
import time
from contextlib import contextmanager
from importlib.metadata import version
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.csv as arrow_csv
import pyarrow.parquet as pq

from .quality import EXPECTED_COLUMNS, normalise_quotes


def _stamp(path):
    stat = Path(path).stat()
    return stat.st_size, stat.st_mtime_ns


def _sha256(path):
    before = _stamp(path)
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    if _stamp(path) != before:
        raise RuntimeError(f"File changed while hashing: {path}")
    return digest.hexdigest()


def _save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="\n") as handle:
        json.dump(value, handle, indent=2, sort_keys=True, allow_nan=False)
        handle.write("\n")
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


@contextmanager
def _writer_lock(directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    with (directory / "writer.lock").open("a+b") as handle:
        handle.seek(0, 2)
        if handle.tell() == 0:
            handle.write(b"0")
            handle.flush()
        handle.seek(0)
        try:
            if os.name == "nt":
                import msvcrt
                msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
            else:
                import fcntl
                fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except OSError as error:
            raise RuntimeError("Another ingestion writer is active.") from error

        try:
            yield
        finally:
            handle.seek(0)
            if os.name == "nt":
                msvcrt.locking(handle.fileno(), msvcrt.LK_UNLCK, 1)
            else:
                fcntl.flock(handle.fileno(), fcntl.LOCK_UN)


def _write_part(frame, output_dir, relative, kind):
    target = output_dir / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + ".tmp")
    table = pa.Table.from_pandas(frame, preserve_index=False)
    pq.write_table(table, temporary, compression="zstd")
    os.replace(temporary, target)
    return {
        "path": relative,
        "kind": kind,
        "rows": len(frame),
        "bytes": target.stat().st_size,
        "sha256": _sha256(target),
    }


def _verify_parts(state):
    totals = {"quotes": 0, "unassigned": 0}
    root = Path(state["output_dir"]).resolve()

    for part in state["parts"]:
        path = (root / part["path"]).resolve()
        if not path.is_relative_to(root):
            raise RuntimeError("Manifest output path escapes its generation.")
        if not path.is_file() or _sha256(path) != part["sha256"]:
            raise RuntimeError(f"Output checksum verification failed: {path}")
        if pq.ParquetFile(path).metadata.num_rows != part["rows"]:
            raise RuntimeError(f"Output row count changed: {path}")
        totals[part["kind"]] += part["rows"]

    if totals["quotes"] != state["rows_in_scope"]:
        raise RuntimeError("Canonical output row counts do not reconcile.")
    if totals["unassigned"] != state["rows_unassigned"]:
        raise RuntimeError("Unassigned output row counts do not reconcile.")


def _update_bounds(state, prefix, timestamps):
    valid = timestamps.dropna()
    if valid.empty:
        return
    for suffix, candidate, choose in (
        ("min", valid.min(), min),
        ("max", valid.max(), max),
    ):
        key = f"{prefix}_{suffix}"
        existing = state.get(key)
        value = candidate if existing is None else choose(
            candidate, pd.Timestamp(existing)
        )
        state[key] = value.isoformat()
'''.removeprefix("\\\n")

print("Ingestion helper contents prepared.")

In [ ]:
data_ingestion = r'''

def ingest_quotes(
    source_path,
    data_root,
    manifest_root,
    start,
    end,
    *,
    timezone,
    timestamp_format,
    pip_size,
    gap_seconds,
    extreme_spread_pips,
    block_bytes,
):
    source = Path(source_path).resolve()
    data_root = Path(data_root).resolve()
    manifest_root = Path(manifest_root).resolve()

    if not source.is_file():
        raise FileNotFoundError(source)

    start, end = pd.Timestamp(start), pd.Timestamp(end)
    if start.tzinfo is None or end.tzinfo is None:
        raise ValueError("Range boundaries must include a timezone.")
    start, end = start.tz_convert("UTC"), end.tz_convert("UTC")
    if start >= end or block_bytes < 128:
        raise ValueError("Invalid time range or CSV block size.")

    source_stamp = _stamp(source)
    source_hash = _sha256(source)

    implementation = hashlib.sha256(
        Path(__file__).read_bytes()
        + Path(__file__).with_name("quality.py").read_bytes()
    ).hexdigest()

    parameters = {
        "timezone": timezone,
        "timestamp_format": timestamp_format,
        "pip_size": pip_size,
        "gap_seconds": gap_seconds,
        "extreme_spread_pips": extreme_spread_pips,
        "block_bytes": block_bytes,
    }
    identity = {
        "source_path": os.path.normcase(str(source)),
        "source_sha256": source_hash,
        "source_bytes": source_stamp[0],
        "start_utc": start.isoformat(),
        "end_utc": end.isoformat(),
        "parameters": parameters,
        "implementation_sha256": implementation,
        "python": platform.python_version(),
        "versions": {
            name: version(name)
            for name in ("numpy", "pandas", "pyarrow", "duckdb", "tzdata")
        },
    }
    job_id = hashlib.sha256(
        json.dumps(identity, sort_keys=True).encode("utf-8")
    ).hexdigest()[:24]

    output_dir = data_root / job_id
    manifest_path = manifest_root / f"job-{job_id}.json"
    registry_key = hashlib.sha256(
        os.path.normcase(str(source)).encode("utf-8")
    ).hexdigest()[:24]
    registry_path = manifest_root / f"source-{registry_key}.json"

    with _writer_lock(manifest_root):
        if registry_path.exists():
            registered = json.loads(registry_path.read_text(encoding="utf-8"))
            if registered["source_sha256"] != source_hash:
                raise ValueError(
                    "Source content changed. Stop and review the replacement "
                    "before combining it with previous research outputs."
                )
        else:
            _save_json(registry_path, {
                "source_path": str(source),
                "source_sha256": source_hash,
                "source_bytes": source_stamp[0],
            })

        if manifest_path.exists():
            state = json.loads(manifest_path.read_text(encoding="utf-8"))
            if state["identity"] != identity:
                raise RuntimeError("Existing manifest identity is incompatible.")
            _verify_parts(state)
            if state["status"] == "complete":
                return {**state, "cache_hit": True}
        else:
            output_dir.mkdir(parents=True, exist_ok=True)
            state = {
                "job_id": job_id,
                "identity": identity,
                "manifest_path": str(manifest_path),
                "output_dir": str(output_dir),
                "status": "running",
                "next_batch": 0,
                "rows_scanned": 0,
                "rows_in_scope": 0,
                "rows_unassigned": 0,
                "rows_outside_scope": 0,
                "previous_utc": None,
                "source_flags": {},
                "scope_flags": {},
                "parts": [],
                "maximum_observed_batch_bytes": 0,
            }
            _save_json(manifest_path, state)

        committed_batches = state["next_batch"]
        row_offset = 0

        with source.open("rb") as handle:
            if handle.read(3) != b"\xef\xbb\xbf":
                handle.seek(0)

            reader = arrow_csv.open_csv(
                handle,
                read_options=arrow_csv.ReadOptions(
                    block_size=block_bytes,
                    use_threads=False,
                ),
                parse_options=arrow_csv.ParseOptions(
                    delimiter=",",
                    newlines_in_values=True,
                    ignore_empty_lines=False,
                ),
                convert_options=arrow_csv.ConvertOptions(
                    column_types={name: pa.string() for name in EXPECTED_COLUMNS},
                    strings_can_be_null=False,
                    check_utf8=True,
                ),
            )
            try:
                if reader.schema.names != EXPECTED_COLUMNS:
                    raise ValueError(
                        f"Unexpected source header: {reader.schema.names}"
                    )

                for batch_number, batch in enumerate(reader):
                    if batch_number < committed_batches:
                        row_offset += batch.num_rows
                        if (
                            batch_number + 1 == committed_batches
                            and row_offset != state["rows_scanned"]
                        ):
                            raise RuntimeError("Checkpoint replay row mismatch.")
                        continue

                    if (
                        batch_number != state["next_batch"]
                        or row_offset != state["rows_scanned"]
                    ):
                        raise RuntimeError("Checkpoint position mismatch.")

                    raw = batch.to_pandas()
                    frame = normalise_quotes(
                        raw,
                        source_name=source.name,
                        source_sha256=source_hash,
                        first_row=row_offset + 1,
                        previous_utc=state["previous_utc"],
                        **{key: value for key, value in parameters.items()
                           if key != "block_bytes"},
                    )

                    unassigned = frame["timestamp_utc"].isna()
                    in_scope = (
                        frame["timestamp_utc"].ge(start)
                        & frame["timestamp_utc"].lt(end)
                    )
                    selected = frame.loc[in_scope]
                    new_parts = []

                    if not selected.empty:
                        years = selected["timestamp_utc"].dt.year
                        months = selected["timestamp_utc"].dt.month
                        for (year, month), group in selected.groupby(
                            [years, months], sort=True
                        ):
                            relative = (
                                f"quotes/year={int(year):04d}/"
                                f"month={int(month):02d}/"
                                f"part-{batch_number:07d}.parquet"
                            )
                            new_parts.append(
                                _write_part(group, output_dir, relative, "quotes")
                            )

                    if unassigned.any():
                        relative = (
                            f"unassigned/part-{batch_number:07d}.parquet"
                        )
                        new_parts.append(_write_part(
                            frame.loc[unassigned],
                            output_dir,
                            relative,
                            "unassigned",
                        ))

                    flags = [name for name in frame if name.startswith("flag_")]
                    for key, subset in (
                        ("source_flags", frame),
                        ("scope_flags", selected),
                    ):
                        for name, count in subset[flags].sum().items():
                            state[key][name] = (
                                state[key].get(name, 0) + int(count)
                            )

                    state["parts"].extend(new_parts)
                    state["rows_scanned"] += len(frame)
                    state["rows_in_scope"] += len(selected)
                    state["rows_unassigned"] += int(unassigned.sum())
                    state["rows_outside_scope"] += (
                        len(frame) - len(selected) - int(unassigned.sum())
                    )
                    _update_bounds(state, "source_utc", frame["timestamp_utc"])
                    _update_bounds(state, "scope_utc", selected["timestamp_utc"])

                    if len(frame):
                        last = frame["timestamp_utc"].iloc[-1]
                        state["previous_utc"] = (
                            None if pd.isna(last) else last.isoformat()
                        )

                    observed_bytes = (
                        batch.nbytes
                        + int(frame.memory_usage(index=True, deep=True).sum())
                        + int(raw.memory_usage(index=True, deep=True).sum())
                    )
                    state["maximum_observed_batch_bytes"] = max(
                        state["maximum_observed_batch_bytes"],
                        observed_bytes,
                    )
                    row_offset += len(frame)
                    state["next_batch"] = batch_number + 1
                    _save_json(manifest_path, state)

                    if state["next_batch"] % 50 == 0:
                        print(
                            f"{source.name}: "
                            f"{state['rows_scanned']:,} records scanned; "
                            f"{state['rows_in_scope']:,} retained"
                        )
            finally:
                reader.close()

        if row_offset != state["rows_scanned"]:
            raise RuntimeError("Final source row count does not reconcile.")

        if (
            state["rows_scanned"]
            != state["rows_in_scope"]
            + state["rows_unassigned"]
            + state["rows_outside_scope"]
        ):
            raise RuntimeError("Source row conservation check failed.")

        if _stamp(source) != source_stamp or _sha256(source) != source_hash:
            raise RuntimeError("Source changed during ingestion; job not completed.")

        _verify_parts(state)
        state["status"] = "complete"
        _save_json(manifest_path, state)
        return {**state, "cache_hit": False}
'''

print("Ingestion function contents prepared.")

In [ ]:
data_diagnostics = r'''

def _quote_files(state):
    return [
        str(Path(state["output_dir"]) / part["path"])
        for part in state["parts"]
        if part["kind"] == "quotes"
    ]


def _quote_connection(state):
    if state["status"] != "complete" or not _quote_files(state):
        raise ValueError("A completed generation with in-range quotes is required.")
    temporary = Path(state["output_dir"]) / "duckdb_tmp"
    temporary.mkdir(parents=True, exist_ok=True)
    connection = duckdb.connect(config={
        "memory_limit": "256MB",
        "threads": 2,
        "temp_directory": str(temporary),
        "max_temp_directory_size": "4GB",
    })
    connection.execute("SET TimeZone = 'UTC'")
    connection.read_parquet(
        _quote_files(state), hive_partitioning=False
    ).create_view("quotes")
    return connection


def summarise_generation(state):
    connection = _quote_connection(state)
    try:
        count, minimum, maximum, repeated = connection.execute("""
            SELECT count(*), min(timestamp_utc), max(timestamp_utc),
                   count(*) - count(DISTINCT timestamp_utc)
            FROM quotes
        """).fetchone()

        duplicate_groups, duplicate_excess = connection.execute("""
            WITH duplicates AS (
                SELECT count(*) AS n
                FROM quotes
                GROUP BY timestamp_source, ask_raw, bid_raw,
                         ask_size_raw, bid_size_raw
                HAVING count(*) > 1
            )
            SELECT count(*), coalesce(sum(n - 1), 0)
            FROM duplicates
        """).fetchone()

        gap_count, maximum_gap = connection.execute("""
            WITH differences AS (
                SELECT epoch(
                    timestamp_utc
                    - lag(timestamp_utc) OVER (
                        ORDER BY timestamp_utc, source_row
                    )
                ) AS gap_seconds
                FROM quotes
            )
            SELECT count(*) FILTER (WHERE gap_seconds > ?),
                   max(gap_seconds)
            FROM differences
        """, [state["identity"]["parameters"]["gap_seconds"]]).fetchone()

        spread_min, spread_median, spread_max = connection.execute("""
            SELECT min(spread_pips),
                   approx_quantile(spread_pips, 0.5),
                   max(spread_pips)
            FROM quotes
            WHERE isfinite(spread_pips)
        """).fetchone()

        flag_names = sorted(state["scope_flags"])
        condition = " OR ".join(flag_names)
        example_columns = [
            "source_row", "timestamp_source", "timestamp_utc",
            "ask", "bid", "ask_size", "bid_size", "spread_pips",
            *flag_names,
        ]
        examples = connection.execute(
            "SELECT " + ", ".join(example_columns)
            + " FROM quotes WHERE " + condition
            + " ORDER BY source_row LIMIT 5"
        ).fetchdf()

        if int(count) != state["rows_in_scope"]:
            raise RuntimeError("DuckDB count disagrees with ingestion manifest.")

        return {
            "rows": int(count),
            "utc_min": minimum.isoformat(),
            "utc_max": maximum.isoformat(),
            "repeated_timestamp_excess_rows": int(repeated),
            "duplicate_field_groups": int(duplicate_groups),
            "duplicate_field_excess_rows": int(duplicate_excess),
            "chronological_gaps_above_threshold": int(gap_count),
            "maximum_chronological_gap_seconds": maximum_gap,
            "spread_pips": {
                "minimum": spread_min,
                "approximate_median": spread_median,
                "maximum": spread_max,
            },
            "scope_flags": state["scope_flags"],
            "flagged_examples": json.loads(
                examples.to_json(orient="records", date_format="iso")
            ),
        }
    finally:
        connection.close()


def compare_day_with_month(day, month, start, end):
    connection = _quote_connection(day)
    try:
        connection.read_parquet(
            _quote_files(month), hive_partitioning=False
        ).create_view("month_quotes")
        difference_rows = connection.execute("""
            WITH month_slice AS (
                SELECT * FROM month_quotes
                WHERE timestamp_utc >= CAST(? AS TIMESTAMPTZ)
                  AND timestamp_utc < CAST(? AS TIMESTAMPTZ)
            ),
            differences AS (
                (SELECT * FROM quotes
                 EXCEPT ALL SELECT * FROM month_slice)
                UNION ALL
                (SELECT * FROM month_slice
                 EXCEPT ALL SELECT * FROM quotes)
            )
            SELECT count(*) FROM differences
        """, [start, end]).fetchone()[0]
        return int(difference_rows)
    finally:
        connection.close()
'''

new_files["src/eurusd_research/data.py"] = (
    data_helpers + data_ingestion + data_diagnostics
)

print("Complete data module contents prepared.")

In [ ]:
new_files["data/sample/synthetic_quotes.csv"] = """\
Time (EET),Ask,Bid,AskVolume,BidVolume
2020.02.01 01:59:59.500,1.1002,1.1000,1,3
2020.02.01 02:00:00.000,1.1003,1.1001,1,3
2020.02.01 02:00:00.000,1.1003,1.1001,1,3
2020.02.01 02:00:00.000,1.1004,1.1001,1,3
2020.02.01 02:00:01.000,1.1000,1.1002,1,3
2020.02.01 02:00:00.500,1.1002,1.1002,1,3
2020.02.01 02:00:45.000,0,1.1000,,0
2020.02.01 02:00:46.000,1.1002,1.1000,-1,2
2020.02.01 02:00:47.000,1.1030,1.1000,1,3
2020.02.01 02:00:48.000,1.1002,1.1000,0,0
2020.02.01 02:00:49.000,bad,1.1000,NaN,inf
not-a-date,1.1002,1.1000,1,3
2020.03.29 03:30:00.000,1.1002,1.1000,1,3
2020.10.25 03:30:00.000,1.1002,1.1000,1,3
"""

new_files["tests/test_alignment.py"] = r'''\
import pandas as pd
import pytest

from eurusd_research.quality import normalise_quotes


def normalise(times, **overrides):
    raw = pd.DataFrame({
        "Time (EET)": times,
        "Ask": ["1.1002"] * len(times),
        "Bid": ["1.1000"] * len(times),
        "AskVolume": ["1"] * len(times),
        "BidVolume": ["3"] * len(times),
    })
    options = {
        "source_name": "synthetic.csv",
        "source_sha256": "test",
        "first_row": 1,
        "previous_utc": None,
        "timezone": "Europe/Helsinki",
        "timestamp_format": "%Y.%m.%d %H:%M:%S.%f",
        "pip_size": 0.0001,
        "gap_seconds": 30,
        "extreme_spread_pips": 20,
    }
    options.update(overrides)
    return normalise_quotes(raw, **options)


@pytest.mark.parametrize("source, expected", [
    ("2025.01.06 00:00:11.712", "2025-01-05T22:00:11.712Z"),
    ("2025.03.09 23:00:09.602", "2025-03-09T21:00:09.602Z"),
    ("2025.03.31 00:00:09.314", "2025-03-30T21:00:09.314Z"),
    ("2025.07.07 00:00:17.843", "2025-07-06T21:00:17.843Z"),
    ("2025.10.26 23:00:03.449", "2025-10-26T21:00:03.449Z"),
])
def test_historical_clock_conversion(source, expected):
    result = normalise([source])
    assert result.loc[0, "timestamp_utc"] == pd.Timestamp(expected)
    assert result.loc[0, "spread_pips"] == pytest.approx(2.0)


def test_dst_times_are_not_guessed():
    result = normalise([
        "2020.03.29 03:30:00.000",
        "2020.10.25 03:30:00.000",
        "not-a-date",
    ])
    assert result["timestamp_utc"].isna().all()
    assert result["flag_unresolved_local_time"].tolist() == [True, True, False]
    assert result["flag_unparseable_timestamp"].tolist() == [False, False, True]


def test_previous_batch_timestamp_is_used():
    result = normalise(
        ["2020.02.01 02:00:00.000"],
        previous_utc="2020-02-01T00:00:01Z",
    )
    assert bool(result.loc[0, "flag_backward_timestamp"])
'''.removeprefix("\\\n")

new_files["tests/test_partition_consistency.py"] = r'''\
from pathlib import Path

import pandas as pd
import pytest

from eurusd_research import data


@pytest.fixture
def source(tmp_path):
    sample = Path(__file__).resolve().parents[1] / "data/sample/synthetic_quotes.csv"
    target = tmp_path / "quotes.csv"
    target.write_bytes(sample.read_bytes())
    return target


def run(source, root, block_bytes=128, start="2020-01-31T00:00:00Z"):
    return data.ingest_quotes(
        source, root / "data", root / "manifests",
        start, "2020-02-02T00:00:00Z",
        timezone="Europe/Helsinki",
        timestamp_format="%Y.%m.%d %H:%M:%S.%f",
        pip_size=0.0001,
        gap_seconds=30,
        extreme_spread_pips=20,
        block_bytes=block_bytes,
    )


def rows(state):
    frames = [
        pd.read_parquet(Path(state["output_dir"]) / part["path"])
        for part in state["parts"] if part["kind"] == "quotes"
    ]
    return pd.concat(frames, ignore_index=True).sort_values(
        "source_row"
    ).reset_index(drop=True)


def test_batch_consistency_and_quality(source, tmp_path):
    small = run(source, tmp_path / "small", 128)
    large = run(source, tmp_path / "large", 65536)
    pd.testing.assert_frame_equal(rows(small), rows(large))

    assert small["rows_scanned"] == 14
    assert small["rows_in_scope"] == 11
    assert small["rows_unassigned"] == 3
    assert small["source_flags"] == large["source_flags"]

    report = data.summarise_generation(small)
    assert report["duplicate_field_excess_rows"] == 1
    assert report["repeated_timestamp_excess_rows"] == 2
    assert report["scope_flags"]["flag_crossed"] == 1
    assert report["scope_flags"]["flag_locked"] == 1
    assert report["scope_flags"]["flag_missing_size"] == 1
    assert report["scope_flags"]["flag_negative_size"] == 1
    assert report["scope_flags"]["flag_zero_total_size"] == 1


def test_completed_rerun_and_changed_source(source, tmp_path):
    root = tmp_path / "run"
    first = run(source, root)
    repeated = run(source, root)
    assert repeated["cache_hit"]
    assert first["manifest_path"] == repeated["manifest_path"]
    pd.testing.assert_frame_equal(rows(first), rows(repeated))

    source.write_bytes(source.read_bytes() + b"\n")
    with pytest.raises(ValueError, match="Source content changed"):
        run(source, root)


def test_damaged_output_is_detected(source, tmp_path):
    root = tmp_path / "run"
    state = run(source, root)
    part = next(part for part in state["parts"] if part["kind"] == "quotes")
    (Path(state["output_dir"]) / part["path"]).write_bytes(b"damaged")
    with pytest.raises(RuntimeError, match="Output checksum"):
        run(source, root)


def test_interrupted_empty_output_batch_resumes(source, tmp_path, monkeypatch):
    original = data._save_json
    interrupted = False

    def save_then_interrupt(path, value):
        nonlocal interrupted
        original(path, value)
        if (
            Path(path).name.startswith("job-")
            and value.get("next_batch") == 1
            and value.get("status") == "running"
            and not interrupted
        ):
            interrupted = True
            raise RuntimeError("simulated interruption")

    monkeypatch.setattr(data, "_save_json", save_then_interrupt)
    root = tmp_path / "resume"

    with pytest.raises(RuntimeError, match="simulated interruption"):
        run(source, root, start="2020-02-01T00:00:00Z")

    resumed = run(source, root, start="2020-02-01T00:00:00Z")
    reference = run(
        source, tmp_path / "reference", 65536,
        start="2020-02-01T00:00:00Z",
    )
    pd.testing.assert_frame_equal(rows(resumed), rows(reference))
'''.removeprefix("\\\n")

print("Synthetic data and test contents prepared.")

In [ ]:
example_config = json.loads(example_path.read_text(encoding="utf-8-sig"))
local_config = json.loads(local_path.read_text(encoding="utf-8-sig"))

if not isinstance(example_config, dict) or not isinstance(local_config, dict):
    raise ValueError("Configuration files must contain JSON objects.")

source_config = example_config.setdefault("source", {})
existing_timezone = source_config.get("source_timezone")

if existing_timezone not in (None, "Europe/Helsinki"):
    raise ValueError(
        f"Unexpected existing source timezone: {existing_timezone}. "
        "Return this output before changing it."
    )

local_source = local_config.get("source", {})
if local_source.get("source_timezone") not in (None, "Europe/Helsinki"):
    raise ValueError("Local configuration overrides the verified source timezone.")

source_config.update({
    "source_timezone": "Europe/Helsinki",
    "source_timezone_verified": True,
    "timezone_validation_scope": (
        "Selected spring/autumn weekend boundaries in "
        "2016, 2019, 2021, 2023 and 2025; full-history checks remain pending."
    ),
})

ingestion_defaults = {
    "source_file": "EURUSD_Ticks_2016.01.04_2017.01.02.csv",
    "day_start_utc": "2016-01-05T00:00:00Z",
    "day_end_utc": "2016-01-06T00:00:00Z",
    "month_start_utc": "2016-01-01T00:00:00Z",
    "month_end_utc": "2016-02-01T00:00:00Z",
    "csv_block_bytes": 4 * 1024 * 1024,
    "gap_audit_seconds": 30,
    "extreme_spread_audit_pips": 20,
}

ingestion_config = example_config.setdefault("ingestion", {})
for key, value in ingestion_defaults.items():
    ingestion_config.setdefault(key, value)

create_missing_files(new_files)

example_path.write_text(
    json.dumps(example_config, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
    newline="\n",
)

(REPO_ROOT / "notebooks").mkdir(exist_ok=True)
print("Shared configuration extended; local configuration preserved.")

In [ ]:
installation = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--editable",
        f"{REPO_ROOT}[test]",
    ],
    check=True,
)

print("Editable installation command completed.")

In [ ]:
probe_code = """
import json
from pathlib import Path
import numpy
import pandas
import pyarrow
import duckdb
import matplotlib
import pytest
import eurusd_research
from eurusd_research import config, data, quality

print(json.dumps({
    "working_directory": str(Path.cwd()),
    "package_file": eurusd_research.__file__,
    "imports": "passed"
}))
"""

probe = subprocess.run(
    [sys.executable, "-c", probe_code],
    cwd=REPO_ROOT / "notebooks",
    capture_output=True,
    text=True,
    check=True,
)

import_probe = json.loads(probe.stdout)
expected_package = REPO_ROOT / "src/eurusd_research/__init__.py"

if Path(import_probe["package_file"]).resolve() != expected_package.resolve():
    raise RuntimeError("Python imported a different copy of the research package.")

from eurusd_research.config import load_config
from eurusd_research.data import (
    compare_day_with_month,
    ingest_quotes,
    summarise_generation,
)

config = load_config(REPO_ROOT)
ingestion = config["ingestion"]

if config["source"]["source_timezone"] != "Europe/Helsinki":
    raise ValueError("This notebook expects the inspected JForex EET exports.")

if config["pip_size"] != 0.0001:
    raise ValueError("EURUSD pip_size must be 0.0001.")

RAW_DIR = Path(config["raw_data_dir"])
SOURCE_FILE = RAW_DIR / ingestion["source_file"]
DATA_ROOT = REPO_ROOT / "data/interim/ingestion"
MANIFEST_ROOT = REPO_ROOT / "artifacts/manifests/ingestion"

if not SOURCE_FILE.is_file():
    raise FileNotFoundError(SOURCE_FILE)

parameters = {
    "timezone": config["source"]["source_timezone"],
    "timestamp_format": config["source"]["timestamp_format"],
    "pip_size": config["pip_size"],
    "gap_seconds": ingestion["gap_audit_seconds"],
    "extreme_spread_pips": ingestion["extreme_spread_audit_pips"],
    "block_bytes": ingestion["csv_block_bytes"],
}

print(json.dumps(import_probe, indent=2))
print("Selected source:", SOURCE_FILE.name)
print("Source bytes:", SOURCE_FILE.stat().st_size)
print("CSV block bytes:", parameters["block_bytes"])

In [ ]:
import importlib
import subprocess
import sys
import tomllib

tests_passed = False

project_file = REPO_ROOT / "pyproject.toml"
project_text = project_file.read_text(encoding="utf-8")
dependencies = tomllib.loads(project_text)["project"]["dependencies"]

if "pytz" not in dependencies:
    anchor = '    "tzdata",\n'

    if project_text.count(anchor) != 1:
        raise RuntimeError(
            "The dependency section differs from the supplied version. "
            "Return this message before editing the file."
        )

    project_text = project_text.replace(
        anchor,
        anchor + '    "pytz",\n',
        1,
    )

    tomllib.loads(project_text)  # Validate before writing.
    project_file.write_text(
        project_text,
        encoding="utf-8",
        newline="\n",
    )

# Keep this kernel's prepared file contents consistent.
if "new_files" in globals():
    new_files["pyproject.toml"] = project_text

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--editable",
        f"{REPO_ROOT}[test]",
    ],
    check=True,
)

importlib.invalidate_caches()
import pytz

print("pytz version:", pytz.__version__)

test_run = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_alignment.py",
        "tests/test_partition_consistency.py",
        "-q",
    ],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)

print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)

if test_run.returncode != 0:
    raise RuntimeError(
        "Tests remain unresolved. Return the complete printed test output."
    )

tests_passed = True
test_summary = test_run.stdout.strip().splitlines()[-1]

In [ ]:
import time

run_measurements = {}


def execute_scope(label, source, start, end):
    started = time.perf_counter()
    state = ingest_quotes(
        source,
        DATA_ROOT,
        MANIFEST_ROOT,
        start,
        end,
        **parameters,
    )
    quality = summarise_generation(state)
    elapsed = time.perf_counter() - started

    run_measurements[label] = {
        "runtime_seconds_this_execution": round(elapsed, 3),
        "parquet_output_bytes": sum(part["bytes"] for part in state["parts"]),
        "maximum_observed_batch_bytes": state["maximum_observed_batch_bytes"],
        "cache_hit": state["cache_hit"],
    }

    print(label.upper())
    print("Source records scanned:", f"{state['rows_scanned']:,}")
    print("In-range quotes written:", f"{state['rows_in_scope']:,}")
    print("Unassignable timestamp rows preserved:", state["rows_unassigned"])
    print("UTC coverage:", quality["utc_min"], "to", quality["utc_max"])
    print("Measurements:", run_measurements[label])
    return state, quality


synthetic_state, synthetic_quality = execute_scope(
    "synthetic",
    REPO_ROOT / "data/sample/synthetic_quotes.csv",
    "2020-01-31T00:00:00Z",
    "2020-02-02T00:00:00Z",
)

assert synthetic_state["rows_scanned"] == 14
assert synthetic_state["rows_in_scope"] == 11
assert synthetic_state["rows_unassigned"] == 3
assert synthetic_quality["duplicate_field_excess_rows"] == 1
assert synthetic_quality["repeated_timestamp_excess_rows"] == 2

synthetic_partitions = {
    str(Path(part["path"]).parent)
    for part in synthetic_state["parts"]
    if part["kind"] == "quotes"
}
assert len(synthetic_partitions) == 2

print("Synthetic row conservation, duplicate handling, and UTC partitions: PASS")

In [ ]:
import shutil

free_gib = shutil.disk_usage(REPO_ROOT).free / (1024 ** 3)
print("Available repository-drive space, GiB:", round(free_gib, 2))

if free_gib < 6:
    raise RuntimeError(
        "Less than 6 GiB is available. Free space before running ingestion "
        "and DuckDB diagnostics."
    )

day_state, day_quality = execute_scope(
    "day",
    SOURCE_FILE,
    ingestion["day_start_utc"],
    ingestion["day_end_utc"],
)

print("Pilot-day quality flags:")
print(json.dumps(day_quality["scope_flags"], indent=2))

In [ ]:
RUN_MONTH_INGESTION = True

if not RUN_MONTH_INGESTION:
    raise RuntimeError(
        "Month ingestion deliberately postponed. Resume at Cell 24 when ready."
    )

free_gib = shutil.disk_usage(REPO_ROOT).free / (1024 ** 3)
if free_gib < 6:
    raise RuntimeError("Free at least 6 GiB before the month run.")

month_state, month_quality = execute_scope(
    "month",
    SOURCE_FILE,
    ingestion["month_start_utc"],
    ingestion["month_end_utc"],
)

day_month_difference_rows = compare_day_with_month(
    day_state,
    month_state,
    ingestion["day_start_utc"],
    ingestion["day_end_utc"],
)

if day_month_difference_rows != 0:
    raise RuntimeError(
        "Day/month observations differ. Stop and return Cell 24 output."
    )

print("Day versus month-slice comparison: PASS")
print("Difference rows:", day_month_difference_rows)
print("Month quality flags:")
print(json.dumps(month_quality["scope_flags"], indent=2))

In [ ]:
from importlib.metadata import PackageNotFoundError, version

assert tests_passed
assert day_month_difference_rows == 0

private_dir = REPO_ROOT / "reports/private"
private_dir.mkdir(parents=True, exist_ok=True)

quality_path = private_dir / f"quality-{month_state['job_id']}.json"
quality_path.write_text(
    json.dumps(
        {
            "synthetic": synthetic_quality,
            "day": day_quality,
            "month": month_quality,
            "measurements": run_measurements,
        },
        indent=2,
        allow_nan=False,
    ) + "\n",
    encoding="utf-8",
)

validated_versions = {}
for name in (
    "numpy", "pandas", "pyarrow", "duckdb", "matplotlib",
    "pytest", "tzdata", "jupyterlab", "ipykernel", "setuptools",
):
    try:
        validated_versions[name] = version(name)
    except PackageNotFoundError:
        validated_versions[name] = None

environment_path = private_dir / "validated_ingestion_environment.json"
environment_path.write_text(
    json.dumps({
        "python": platform.python_version(),
        "versions": validated_versions,
        "validation_scope": "Notebook 01 imports, tests, and ingestion",
        "modeling_and_forecast_evaluation": "pending",
    }, indent=2) + "\n",
    encoding="utf-8",
)

updated_local = json.loads(local_path.read_text(encoding="utf-8-sig"))
updated_local["active_ingestion_manifest"] = str(
    Path(month_state["manifest_path"]).relative_to(REPO_ROOT)
).replace("\\", "/")
local_path.write_text(
    json.dumps(updated_local, indent=2) + "\n",
    encoding="utf-8",
    newline="\n",
)


def marked_update(path, tag, content):
    existing = path.read_text(encoding="utf-8-sig") if path.exists() else ""
    beginning = f"<!-- BEGIN {tag} -->"
    ending = f"<!-- END {tag} -->"
    block = beginning + "\n" + content.rstrip() + "\n" + ending

    if beginning in existing or ending in existing:
        if existing.count(beginning) != 1 or existing.count(ending) != 1:
            raise ValueError(f"Invalid generated-section markers in {path.name}")
        start = existing.index(beginning)
        finish = existing.index(ending, start) + len(ending)
        result = existing[:start] + block + existing[finish:]
    else:
        result = existing.rstrip() + "\n\n" + block + "\n"

    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(result, encoding="utf-8", newline="\n")


quality_text = f"""\
## Notebook 01 — ingestion execution

Source timezone: Europe/Helsinki, interpreted as European EET/EEST.
Timezone evidence: selected historical DST weekend boundaries.

Requested month: {ingestion['month_start_utc']} to
{ingestion['month_end_utc']} (exclusive end).

Observed month coverage: {month_quality['utc_min']} to
{month_quality['utc_max']}.

- Source records scanned: {month_state['rows_scanned']:,}
- In-range quotes preserved: {month_state['rows_in_scope']:,}
- Unassignable timestamp rows preserved separately:
  {month_state['rows_unassigned']:,}
- Valid-timestamp records outside this range:
  {month_state['rows_outside_scope']:,}
- Exact-field duplicate excess rows:
  {month_quality['duplicate_field_excess_rows']:,}
- Repeated-timestamp excess rows:
  {month_quality['repeated_timestamp_excess_rows']:,}
- Chronological gaps above the configured audit threshold:
  {month_quality['chronological_gaps_above_threshold']:,}
- Day versus month-slice difference rows: {day_month_difference_rows}

### In-range quality flags

| Flag | Rows |
|---|---:|
"""
quality_text += "\n".join(
    f"| {name} | {count:,} |"
    for name, count in sorted(month_quality["scope_flags"].items())
)
quality_text += """

Flags overlap and do not constitute an automatic exclusion policy.
No price- or size-based rows were silently removed.

Detailed examples and source-wide counts are retained in ignored local
diagnostics and manifests.

Displayed-size units and true side-specific quote ages remain unresolved.
This execution does not establish full-history data quality or predictive value.
"""

marked_update(
    REPO_ROOT / "reports/data_quality.md",
    "NOTEBOOK01",
    quality_text,
)

marked_update(
    REPO_ROOT / "README.md",
    "NOTEBOOK01",
    """\
## Implemented ingestion workflow

The project now provides an editable src-layout package, synthetic
quotes, bounded CSV-to-Parquet ingestion, UTC year/month partitions,
source provenance, resumable batch manifests, and quality diagnostics.

From an active Jupyter kernel, installation uses its Python executable
with `-m pip install --editable <repository>[test]`.

Notebook 01 runs focused tests, a synthetic example, a pilot day, and
the available January 2016 data. Later notebooks consume the selected
manifest recorded in ignored local configuration.

Historical quotes are preserved with quality flags. Unassignable
timestamps are stored separately. Overlapping pilot and month datasets
must not be concatenated.

Environment versions are recorded locally after successful execution.
Owner review remains pending.

Results pending execution and validation.
""",
)

marked_update(
    REPO_ROOT / "CONTRIBUTIONS.md",
    "NOTEBOOK01",
    """\
## Notebook 01 implementation and execution record

OpenAI Codex assisted with package setup, ingestion, quality flags,
synthetic examples, correctness tests, and documentation.

Raymond Fendi supplied the research objectives, source data, and
platform timezone information.

Read-only historical boundary inspection supported European EET/EEST
for the inspected exports.

This generated section is written only after the notebook's automated
checks and ingestion cells complete successfully. Their detailed
evidence is stored in ignored local artifacts.

Raymond Fendi's review and understanding confirmation remain pending.
No forecasting, final-test evaluation, or profitability finding is claimed.
""",
)

print("Detailed diagnostics:", quality_path)
print("Environment record:", environment_path)
print("Active manifest:", updated_local["active_ingestion_manifest"])
print("Documentation updated; owner review remains pending.")

In [ ]:
def compact_scope(state, quality, label):
    return {
        "requested_start_utc": state["identity"]["start_utc"],
        "requested_end_utc_exclusive": state["identity"]["end_utc"],
        "source_rows_scanned": state["rows_scanned"],
        "output_quote_rows": state["rows_in_scope"],
        "unassignable_timestamp_rows_preserved": state["rows_unassigned"],
        "out_of_scope_rows": state["rows_outside_scope"],
        "observed_utc_min": quality["utc_min"],
        "observed_utc_max": quality["utc_max"],
        "duplicate_field_excess_rows": quality["duplicate_field_excess_rows"],
        "repeated_timestamp_excess_rows": quality["repeated_timestamp_excess_rows"],
        "chronological_gaps_above_threshold": (
            quality["chronological_gaps_above_threshold"]
        ),
        "maximum_chronological_gap_seconds": (
            quality["maximum_chronological_gap_seconds"]
        ),
        "spread_pips": quality["spread_pips"],
        "scope_flags": quality["scope_flags"],
        "source_wide_flags": state["source_flags"],
        "measurements": run_measurements[label],
        "manifest": str(
            Path(state["manifest_path"]).relative_to(REPO_ROOT)
        ),
    }


summary = {
    "notebook": "01_ingestion_and_quality.ipynb",
    "python_version": platform.python_version(),
    "validated_package_versions": validated_versions,
    "configuration": {
        "source_file": SOURCE_FILE.name,
        **parameters,
    },
    "validation": {
        "imports_from_notebooks_directory": import_probe["imports"],
        "pytest": test_summary,
        "synthetic_checks": "passed",
        "day_month_difference_rows": day_month_difference_rows,
        "source_and_output_integrity": "passed during ingestion",
        "owner_review": "pending",
    },
    "synthetic": compact_scope(
        synthetic_state, synthetic_quality, "synthetic"
    ),
    "day": compact_scope(day_state, day_quality, "day"),
    "month": compact_scope(month_state, month_quality, "month"),
    "warnings": [
        "Displayed-size denomination remains unresolved.",
        "True side-specific quote ages are unavailable.",
        "Full-history quality and cross-file overlaps remain unassessed.",
        "Quality flags overlap and are not automatic exclusion decisions.",
        "Maximum batch bytes are not total process peak memory.",
        "No predictive or profitability finding has been established.",
    ],
    "saved_outputs": {
        "active_manifest": updated_local["active_ingestion_manifest"],
        "detailed_quality": str(quality_path.relative_to(REPO_ROOT)),
        "validated_environment": str(environment_path.relative_to(REPO_ROOT)),
        "public_quality_report": "reports/data_quality.md",
        "documentation": ["README.md", "CONTRIBUTIONS.md"],
    },
}

print("NOTEBOOK_01_SUMMARY")
print(json.dumps(summary, indent=2, allow_nan=False))